- Imports

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from scipy.ndimage import label, binary_dilation
from src import src, misc 

print(tf.config.list_physical_devices('GPU')) 

## imaging forward model

In [ ]:
import os
import numpy as np
import tensorflow as tf
from scipy.ndimage import label, binary_dilation

# ============================================================
# Shared raw-data loader for the same 10 files
# ============================================================
def load_raw_by_select(raw_select, data_dir="data", aug_prefix="aug"):
    """
    Mapping:
      1 -> y_exp_test_4.npy
      2 -> aug_1.npy
      3 -> aug_2.npy
      ...
      10 -> aug_9.npy

    Returns:
      y_tf  : (1,K,T,R) tf.complex64
      y_np  : (K,T,R) np.complex64
      path  : file path
    """
    raw_select = int(raw_select)

    if raw_select == 1:
        path = os.path.join(data_dir, "y_exp_test_4.npy")
    elif 2 <= raw_select <= 10:
        path = os.path.join(data_dir, f"{aug_prefix}_{raw_select - 1}.npy")
    else:
        raise ValueError("raw_select must be between 1 and 10")

    if not os.path.isfile(path):
        raise FileNotFoundError(f"Raw file not found: {path}")

    y_np = np.load(path).astype(np.complex64)              # (K,T,R)
    y_tf = tf.constant(y_np[None, ...], dtype=tf.complex64)  # (1,K,T,R)
    return y_tf, y_np, path


# ============================================================
# Model pipeline
# ============================================================
def get_model_pipeline(model_name):
    if model_name == 'Deep2S':
        dnn = src.get_UNet3D(input_shape=(X, Y, Z, 1))
        dnn.load_weights("models_weights/experimental/Deep2S/model_Nf15_SNR30_exp.h5")

        def forward(y_batch, A_conj):
            AHy = tf.einsum('ktrxyz,Nktr->Nxyz', A_conj, y_batch)
            mag = tf.abs(AHy)
            max_mag = tf.reduce_max(mag, axis=(1, 2, 3), keepdims=True)
            mag_norm = mag / (max_mag + 1e-12)
            return dnn(mag_norm[..., None], training=False)

    elif model_name == 'CV-Deep2S':
        dnn = src.get_CV_UNet(input_shape=(X, Y, Z, 2))
        dnn.load_weights("models_weights/experimental/CV-Deep2S/model_Nf15_SNR30_CV_exp.h5")

        def forward(y_batch, A_conj):
            AHy = tf.einsum('ktrxyz,Nktr->Nxyz', A_conj, y_batch)
            max_mag = tf.reduce_max(tf.abs(AHy), axis=(1, 2, 3), keepdims=True)
            AHy_norm = AHy / tf.cast(max_mag + 1e-12, tf.complex64)
            AHy_ri = tf.stack([tf.math.real(AHy_norm), tf.math.imag(AHy_norm)], axis=-1)
            return dnn(AHy_ri, training=False)

    elif model_name == 'Deep2SP+':
        dnn = tf.keras.models.load_model(
            "models_weights/experimental/Deep2S-Plus/model_Nf15_SNR30_Deep2SP_exp",
            compile=False
        )

        def forward(y_batch, A_conj=None):
            y_ri = tf.stack([tf.math.real(y_batch), tf.math.imag(y_batch)], axis=-1)
            return dnn(y_ri, training=False)

    else:
        raise ValueError(f"Unknown MODEL_TYPE: {model_name}")

    return dnn, forward


# ============================================================
# Target generation
# ============================================================
def generate_target_img(
    clean_img,
    victim_forward,
    sarRawData,                 # victim raw: (1,K,T,R)
    A_conj_tf,
    attack_mode="Full",         # "Full" or "ROI"
    target_mode="noise",        # "noise" or "object"
    seed=42,
    targetRawData=None,         # (1,K,T,R), only for object mode
):
    c = clean_img[..., 0] if len(clean_img.shape) == 5 else clean_img   # (1,X,Y,Z)
    Zc = int(c.shape[3])

    # ----------------------------
    # Build target
    # ----------------------------
    if target_mode.lower() == "noise":
        # shuffle victim raw
        y = sarRawData[0].numpy()               # (K,T,R)
        K, T, R = y.shape
        Np = T * R

        Xv = y.reshape(K, Np)
        rng = np.random.default_rng(seed)
        for k in range(K):
            Xv[k, :] = Xv[k, rng.permutation(Np)]

        y_shuf = tf.constant(Xv.reshape(K, T, R)[None, ...], dtype=tf.complex64)
        target_img = victim_forward(y_shuf, A_conj_tf)
        t = target_img[..., 0] if len(target_img.shape) == 5 else target_img

    elif target_mode.lower() == "object":
        if targetRawData is None:
            raise ValueError("For target_mode='object', pass targetRawData from the same 10-file loader.")

        target_img = victim_forward(targetRawData, A_conj_tf)
        t = target_img[..., 0] if len(target_img.shape) == 5 else target_img

        # optional scale match
        clean_mip = tf.reduce_max(c[0], axis=2).numpy().astype(np.float32)
        targ_mip  = tf.reduce_max(t[0], axis=2).numpy().astype(np.float32)

        s_clean = np.quantile(np.abs(clean_mip).reshape(-1), 0.995) + 1e-12
        s_targ  = np.quantile(np.abs(targ_mip).reshape(-1), 0.995) + 1e-12

        t = tf.cast(t, tf.float32) * tf.constant((s_clean / s_targ), tf.float32)

    else:
        raise ValueError("target_mode must be 'noise' or 'object'")

    # ----------------------------
    # ROI mask from CLEAN image
    # ----------------------------
    roi_mask = None
    if attack_mode == "ROI":
        c_np = c.numpy()[0]                     # (X,Y,Z)
        clean_mip = c_np.max(axis=2)

        thresh = 0.7 * clean_mip.max()
        binary = clean_mip > thresh

        labels, num_features = label(binary)
        if num_features > 0:
            largest_label = np.argmax(np.bincount(labels.ravel())[1:]) + 1
            roi_2d = binary_dilation(labels == largest_label, iterations=1)
            roi_3d = np.repeat(roi_2d[..., None], Zc, axis=2)
            roi_mask = tf.constant(roi_3d, dtype=tf.bool)

    return t, roi_mask

## dia optimization

In [ ]:
import os
import math
import numpy as np
import tensorflow as tf

# ============================================================
# UPDATED CONFIG + TARGET BUILD + DIA LOOP (TF, Deep models)
#  - keeps your existing structure
#  - fixes Pa/Pr power math (use complex norm^2, no abs-norm)
#  - adds WHILE-loop, post-projection loss eval, best tracking,
#    and plateau early stop (like your PyTorch "deep models" loop)
# ============================================================

# ------------------ config
MODEL_TYPE        = "CV-Deep2S"     # 'Deep2S', 'CV-Deep2S', 'Deep2SP+'
ATTACK_MODE       = "Full"       # 'Full' or 'ROI'
TARGET_MODE       = "object"     # 'noise' or 'object'

# ---- victim raw data selection (1-based)
VICTIM_RAW_SELECT =  1         # 1 -> y_exp_test_4.npy, 2 -> aug_1.npy, ... 10 -> aug_9.npy

# ---- target raw data selection (used only when TARGET_MODE="object")
TARGET_RAW_SELECT = 2            # same 1..10 mapping as victim
DATA_DIR          = "data"
AUG_PREFIX        = "aug"

# ============================================================
# Constraints / power accounting
# ============================================================
use_amax_projection  = False
use_pa_pr_projection = True

PaPr_max_dB = -10.0
PaPr_max    = 10 ** (PaPr_max_dB / 10.0)


tf.random.set_seed(0)
np.random.seed(0)

# ---- Attack hyperparameters
CONFIG_MAP = {
    'Deep2S': {
        'Full': {'Amax': 2.0, 'lambda_L2': 1e-5, 'max_iter': 1000, 'lr_init': 1e-1,  'lr_switch_iter': 1000000,  'low_lr': 1e-2},
    },
    'CV-Deep2S': {
        'Full': {'Amax': 2.0, 'lambda_L2': 1e-5, 'max_iter': 1000, 'lr_init': 1e-1,  'lr_switch_iter': 1000000, 'low_lr': 1e-2},
    },
    'Deep2SP+': {
        'Full': {'Amax': 2.0, 'lambda_L2': 1e-5, 'max_iter': 1000, 'lr_init': 1e-1,  'lr_switch_iter': 1000000, 'low_lr': 1e-2},
    }
}

In [ ]:
# ------------------ constants
EPS = 1e-12

# ============================================================
# Load A operator + data
# ============================================================
A = np.load("models_weights/experimental/A15_exp.npy")
K, T, R, X, Y, Z = A.shape
A_conj_tf = tf.math.conj(tf.constant(A.astype(np.complex64)))

# ---- victim raw
sarRawData, sarRawData_temp, victim_path = load_raw_by_select(
    raw_select=VICTIM_RAW_SELECT,
    data_dir=DATA_DIR,
    aug_prefix=AUG_PREFIX
)
print(f"[VICTIM] {victim_path}")

# D is (K,TR) complex64, X_v is (K,TR) complex64
D   = tf.constant(np.load(os.path.join(DATA_DIR, "D_flat.npy")).astype(np.complex64))
X_v = tf.constant(sarRawData_temp.reshape(K, T * R), dtype=tf.complex64)

dnn_model, victim_forward = get_model_pipeline(MODEL_TYPE)

# ---- clean image
clean_img = victim_forward(sarRawData, A_conj_tf)
c = clean_img[..., 0] if len(clean_img.shape) == 5 else clean_img
global_scale = tf.reduce_max(tf.abs(c)) + tf.constant(EPS, tf.float32)

# ---- target raw (only for object mode)
targetRawData = None
target_path = None
if TARGET_MODE.lower() == "object":
    targetRawData, _, target_path = load_raw_by_select(
        raw_select=TARGET_RAW_SELECT,
        data_dir=DATA_DIR,
        aug_prefix=AUG_PREFIX
    )
    print(f"[TARGET] {target_path}")

# ---- target image + roi mask
target_img, roi_mask = generate_target_img(
    clean_img=clean_img,
    victim_forward=victim_forward,
    sarRawData=sarRawData,         # used in noise mode
    targetRawData=targetRawData,   # used in object mode
    A_conj_tf=A_conj_tf,
    attack_mode=ATTACK_MODE,
    target_mode=TARGET_MODE,
    seed=42,
)

# ---- ensure target has channel dim if clean has it
if (len(clean_img.shape) == 5) and (len(target_img.shape) == 4):
    target_img = target_img[..., None]   # (1,X,Y,Z) -> (1,X,Y,Z,1)

# ============================================================
# Constraints / power accounting
# ============================================================
#use_amax_projection  = False
#use_pa_pr_projection = False

#PaPr_max_dB = -10.0
#PaPr_max    = 10 ** (PaPr_max_dB / 10.0)

# ============================================================
# Hyperparameters
# ============================================================
params = dict(CONFIG_MAP[MODEL_TYPE][ATTACK_MODE])
Amax           = float(params["Amax"])
lambda_L2      = float(params["lambda_L2"])
max_iter       = int(params["max_iter"])
lr_init        = float(params["lr_init"])
lr_switch_iter = int(params["lr_switch_iter"])
low_lr         = float(params["low_lr"])

# decision vars
A_re = tf.Variable(1e-3 * tf.random.normal(shape=(T * R,), dtype=tf.float32))
A_im = tf.Variable(1e-3 * tf.random.normal(shape=(T * R,), dtype=tf.float32))

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_init)

# ============================================================
# Reference victim power: Pr = ||X_v||^2  (complex Frobenius norm squared)
# ============================================================
Pr_ref = float((tf.linalg.norm(X_v) ** 2).numpy()) + EPS

# ============================================================
# Windowed diminishing-returns stop (reference-style)
# ============================================================
maxIter_cap  = max_iter
check_every  = 25
W_win        = 200
rel_drop_min = 1e-2
abs_drop_min = 0.0
K_weak       = 3
weakWinCount = 0

# ============================================================
# Helper: evaluate feasible iterate
# ============================================================
def eval_postproj(A_complex):
    A_row = tf.reshape(A_complex, (1, T * R))              # (1,TR)
    delta = D * A_row                                      # (K,TR)
    Y_att = tf.reshape(X_v + delta, (1, K, T, R))          # (1,K,T,R)

    adv_img = victim_forward(Y_att, A_conj_tf)

    # shared scaling
    adv_s = adv_img / global_scale
    tgt_s = target_img / global_scale
    cln_s = clean_img / global_scale

    if ATTACK_MODE == "ROI":
        diff_AT = tf.boolean_mask(adv_s[0] - tgt_s[0], roi_mask)
        mse_AT  = tf.reduce_mean(diff_AT**2)

        diff_CA = tf.boolean_mask(adv_s[0] - cln_s[0], roi_mask)
        mse_CA  = tf.reduce_mean(diff_CA**2)
    else:
        mse_AT = tf.reduce_mean((adv_s - tgt_s)**2)
        mse_CA = tf.reduce_mean((adv_s - cln_s)**2)

    reg = lambda_L2 * tf.reduce_mean(tf.abs(A_complex)**2)
    loss_post = mse_AT + reg

    return loss_post, mse_AT, mse_CA

# ============================================================
# DIA LOOP
#   - unconstrained step
#   - projection on updated A
#   - post-projection evaluation
#   - best feasible tracking
#   - diminishing-returns stop
# ============================================================
if TARGET_MODE.lower() == "object":
    tgt_str = f"TARGET_RAW_SELECT={TARGET_RAW_SELECT}"
else:
    tgt_str = "TARGET=SHUFFLED_NOISE"

print(
    f"Starting {ATTACK_MODE} attack on {MODEL_TYPE} ({tgt_str}) | "
    f"lr_init={lr_init:.2e}, lr_switch_iter={lr_switch_iter}, low_lr={low_lr:.2e} | "
    f"Pa/Pr<={PaPr_max_dB:.2f} dB ({int(use_pa_pr_projection)}) | "
    f"|A|<={Amax:.3g} ({int(use_amax_projection)})"
)

bestLoss = float("inf")
bestIter = 0

loss_hist   = []
meanA_hist  = []
maxA_hist   = []
PaPr_hist   = []
PaPrdB_hist = []

A_re_best = tf.identity(A_re)
A_im_best = tf.identity(A_im)

it = 0
while True:
    if it >= maxIter_cap:
        print(f"Reached maxIter cap ({maxIter_cap}).")
        break

    it += 1

    # LR schedule
    current_lr = lr_init if it <= lr_switch_iter else low_lr
    optimizer.learning_rate.assign(current_lr)

    # --------------------------------------------------------
    # 1) Forward / gradients at current A
    # --------------------------------------------------------
    with tf.GradientTape() as tape:
        A_raw = tf.complex(A_re, A_im)                     # (TR,)
        A_row = tf.reshape(A_raw, (1, T * R))             # (1,TR)
        delta = D * A_row                                  # (K,TR)
        Y_att = tf.reshape(X_v + delta, (1, K, T, R))     # (1,K,T,R)

        adv_img = victim_forward(Y_att, A_conj_tf)

        adv_s = adv_img / global_scale
        tgt_s = target_img / global_scale

        if ATTACK_MODE == "ROI":
            diff_AT = tf.boolean_mask(adv_s[0] - tgt_s[0], roi_mask)
            mse_AT = tf.reduce_mean(diff_AT**2)
        else:
            mse_AT = tf.reduce_mean((adv_s - tgt_s)**2)

        reg = lambda_L2 * tf.reduce_mean(tf.abs(A_raw)**2)
        loss = mse_AT + reg

    grads = tape.gradient(loss, [A_re, A_im])
    optimizer.apply_gradients(zip(grads, [A_re, A_im]))

    # --------------------------------------------------------
    # 2) Projection(s) on UPDATED A
    # --------------------------------------------------------
    A_num = tf.complex(A_re, A_im)

    # (a) |A| <= Amax
    if use_amax_projection:
        mags = tf.abs(A_num)
        scale_amax = tf.minimum(1.0, Amax / (mags + EPS))
        A_num = A_num * tf.cast(scale_amax, tf.complex64)

    # (b) Pa/Pr <= PaPr_max
    delta_now = D * tf.reshape(A_num, (1, T * R))
    Pa_now = float((tf.linalg.norm(delta_now) ** 2).numpy()) + EPS
    PaPr_now = Pa_now / Pr_ref

    if use_pa_pr_projection and (PaPr_now > PaPr_max):
        scale_pow = math.sqrt(PaPr_max / (PaPr_now + EPS))
        A_num = A_num * tf.cast(scale_pow, tf.complex64)

        delta_now = D * tf.reshape(A_num, (1, T * R))
        Pa_now = float((tf.linalg.norm(delta_now) ** 2).numpy()) + EPS
        PaPr_now = Pa_now / Pr_ref

    PaPr_now_dB = 10.0 * math.log10(PaPr_now + EPS)

    # write back projected A
    A_re.assign(tf.math.real(A_num))
    A_im.assign(tf.math.imag(A_num))

    meanA_now = float(tf.reduce_mean(tf.abs(A_num)).numpy())
    maxA_now  = float(tf.reduce_max(tf.abs(A_num)).numpy())

    # --------------------------------------------------------
    # 3) Evaluate POST-projection loss (feasible iterate)
    # --------------------------------------------------------
    loss_post, mse_AT_post, mse_CA_post = eval_postproj(A_num)

    lossVal = float(loss_post.numpy())
    mse_ATv = float(mse_AT_post.numpy())
    mse_CAv = float(mse_CA_post.numpy())

    g_re = 0.0 if grads[0] is None else float(tf.reduce_max(tf.abs(grads[0])).numpy())
    g_im = 0.0 if grads[1] is None else float(tf.reduce_max(tf.abs(grads[1])).numpy())
    Gnow = max(g_re, g_im)

    # --------------------------------------------------------
    # 4) Track stats
    # --------------------------------------------------------
    loss_hist.append(lossVal)
    meanA_hist.append(meanA_now)
    maxA_hist.append(maxA_now)
    PaPr_hist.append(PaPr_now)
    PaPrdB_hist.append(PaPr_now_dB)

    # --------------------------------------------------------
    # 5) Update best
    # --------------------------------------------------------
    if lossVal < bestLoss:
        bestLoss = lossVal
        bestIter = it
        A_re_best = tf.identity(A_re)
        A_im_best = tf.identity(A_im)

    # --------------------------------------------------------
    # 6) Logging
    # --------------------------------------------------------
    if (it % 25 == 0) or (it == 1) or (it == maxIter_cap):
        print(
            f"Iter {it:04d}/{maxIter_cap:04d} | "
            f"Loss={lossVal:.4e} (best={bestLoss:.4e} @{bestIter}) | "
            f"G={Gnow:.6e} | "
            f"MSE(A,T)={mse_ATv:.4e}, MSE(C,A)={mse_CAv:.4e} | "
            f"E|A|={meanA_now:.4e}, max|A|={maxA_now:.4e} | "
            f"Pa/Pr={PaPr_now:.4e} ({PaPr_now_dB:.2f} dB)"
        )

    # --------------------------------------------------------
    # 7) Early stop: diminishing returns over a window
    # --------------------------------------------------------
    if it > W_win and (it % check_every == 0):
        L_old = loss_hist[it - W_win - 1]
        L_new = loss_hist[it - 1]

        rel_drop = (L_old - L_new) / max(abs(L_old), 1e-12)
        abs_drop = (L_old - L_new)

        if abs_drop_min > 0:
            isWeak = (rel_drop < rel_drop_min) and (abs_drop < abs_drop_min)
        else:
            isWeak = (rel_drop < rel_drop_min)

        if isWeak:
            weakWinCount += 1
        else:
            weakWinCount = 0

        if weakWinCount >= K_weak:
            print(
                f"Early stop: diminishing returns. Over last {W_win} iters: "
                f"rel_drop={rel_drop:.3e}, abs_drop={abs_drop:.3e}. "
                f"Triggered {weakWinCount}/{K_weak}."
            )
            break

# --------------------------------------------------------
# Convert histories to np arrays
# --------------------------------------------------------
loss_hist   = np.array(loss_hist)
meanA_hist  = np.array(meanA_hist)
maxA_hist   = np.array(maxA_hist)
PaPr_hist   = np.array(PaPr_hist)
PaPrdB_hist = np.array(PaPrdB_hist)

# --------------------------------------------------------
# Restore best A
# --------------------------------------------------------
A_re.assign(A_re_best)
A_im.assign(A_im_best)

print("-----------------------------")
print(f"Attack optimization finished. BestLoss={bestLoss:.4e} at iter {bestIter}. Final iter={it}.")
print("-----------------------------")


## evaluation

In [ ]:
import numpy as np
from matplotlib.patches import Rectangle
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

EPS = 1e-12

def get_roi_box(img2d, thr_frac=0.25, pad=6):
    """
    img2d: (H,W) numpy array
    returns: (r1, r2, c1, c2), inclusive
    """
    A = np.abs(img2d).astype(np.float64)
    A = A / (A.max() + EPS)

    rows = np.where(np.max(A, axis=1) > thr_frac)[0]
    cols = np.where(np.max(A, axis=0) > thr_frac)[0]

    if rows.size == 0 or cols.size == 0:
        r, c = np.unravel_index(np.argmax(A), A.shape)
        rows = np.array([r], dtype=int)
        cols = np.array([c], dtype=int)

    r1 = max(int(rows[0])  - pad, 0)
    r2 = min(int(rows[-1]) + pad, A.shape[0] - 1)
    c1 = max(int(cols[0])  - pad, 0)
    c2 = min(int(cols[-1]) + pad, A.shape[1] - 1)

    return (r1, r2, c1, c2)


def crop_box(img2d, box):
    r1, r2, c1, c2 = box
    return img2d[r1:r2+1, c1:c2+1]


def compute_metrics_roi(x, y):
    x = x.astype(np.float64)
    y = y.astype(np.float64)

    mse = np.mean((x - y) ** 2)

    num = np.sum(x * y)
    den = np.sqrt(np.sum(x**2) * np.sum(y**2)) + EPS
    ncc = num / den

    dr = float(y.max() - y.min()) + EPS

    if x.shape == y.shape and min(x.shape) >= 11:
        ssim_val = ssim(x, y, data_range=dr)
    else:
        ssim_val = np.nan

    if x.shape == y.shape:
        psnr_val = psnr(y, x, data_range=dr)
    else:
        psnr_val = 10.0 * np.log10((dr**2) / (mse + EPS))

    return {
        "mse": mse,
        "ncc": ncc,
        "ssim": ssim_val,
        "psnr": psnr_val,
    }


def draw_roi_box(ax, box, color="r", lw=2):
    r1, r2, c1, c2 = box
    rect = Rectangle(
        (c1, r1),
        c2 - c1 + 1,
        r2 - r1 + 1,
        fill=False,
        edgecolor=color,
        linewidth=lw
    )
    ax.add_patch(rect)

In [ ]:
import math
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

EPS = 1e-12
ORIGIN = "upper"   # keep your current display convention

# ------------------------------------------------------------
# 1) Build attacked measurement + reconstruct attacked volume
# ------------------------------------------------------------
A_opt_tf  = tf.complex(A_re, A_im)                           # (TR,)
A_opt_row = tf.reshape(A_opt_tf, (1, T * R))                 # (1,TR)

delta_opt = D * A_opt_row                                    # (K,TR) complex
Y_opt_tf  = tf.reshape(X_v + delta_opt, (1, K, T, R))        # (1,K,T,R)

attacked_vol = victim_forward(Y_opt_tf, A_conj_tf)           # (1,X,Y,Z,1) or (1,X,Y,Z)

# ------------------------------------------------------------
# 2) Use consistent scaling across attacked/clean/target
# ------------------------------------------------------------
attacked_vol = attacked_vol
clean_vol    = clean_img
target_vol   = target_img

# ------------------------------------------------------------
# 3) Drop channel -> force (1,X,Y,Z)
# ------------------------------------------------------------
adv4 = attacked_vol[..., 0] if len(attacked_vol.shape) == 5 else attacked_vol
cln4 = clean_vol[..., 0]    if len(clean_vol.shape)    == 5 else clean_vol

if tf.is_tensor(target_vol):
    tgt4 = target_vol[..., 0] if len(target_vol.shape) == 5 else target_vol
else:
    tgt4 = tf.constant(np.asarray(target_vol), dtype=tf.float32)

if len(tgt4.shape) == 3:
    tgt4 = tgt4[None, ...]
elif len(tgt4.shape) != 4:
    raise ValueError(f"Unexpected target shape: {tgt4.shape}")

# ------------------------------------------------------------
# 4) MIP (X,Y) for metrics/plots
# ------------------------------------------------------------
adv2d = tf.reduce_max(adv4[0], axis=2).numpy().astype(np.float32)  # (X,Y)
cln2d = tf.reduce_max(cln4[0], axis=2).numpy().astype(np.float32)  # (X,Y)
tgt2d = tf.reduce_max(tgt4[0], axis=2).numpy().astype(np.float32)  # (X,Y)

# ------------------------------------------------------------
# 5) ROI setup for object-mode evaluation
# ------------------------------------------------------------
roi_thr = 0.1
roi_pad = 6

if TARGET_MODE.lower() == "object":
    roi_box_target = get_roi_box(tgt2d, thr_frac=roi_thr, pad=roi_pad)
    roi_box_clean  = get_roi_box(cln2d, thr_frac=roi_thr, pad=roi_pad)

    print(f"Target ROI [r1 r2 c1 c2] = {roi_box_target}")
    print(f"Clean  ROI [r1 r2 c1 c2] = {roi_box_clean}")
else:
    roi_box_target = (0, tgt2d.shape[0]-1, 0, tgt2d.shape[1]-1)
    roi_box_clean  = (0, cln2d.shape[0]-1, 0, cln2d.shape[1]-1)

# ------------------------------------------------------------
# 6) ROI crops for evaluation
# ------------------------------------------------------------
adv_t = crop_box(adv2d, roi_box_target)
tgt_t = crop_box(tgt2d, roi_box_target)

adv_c = crop_box(adv2d, roi_box_clean)
cln_c = crop_box(cln2d, roi_box_clean)

AT = compute_metrics_roi(adv_t, tgt_t)
AC = compute_metrics_roi(adv_c, cln_c)

mse_AT  = AT["mse"]
ncc_AT  = AT["ncc"]
ssim_AT = AT["ssim"]
psnr_AT = AT["psnr"]

mse_AC  = AC["mse"]
ncc_AC  = AC["ncc"]
ssim_AC = AC["ssim"]
psnr_AC = AC["psnr"]

# ------------------------------------------------------------
# 7) Pa/Pr in measurement domain
# ------------------------------------------------------------
Pa = float((tf.linalg.norm(delta_opt) ** 2).numpy()) + EPS
Pr = float((tf.linalg.norm(X_v) ** 2).numpy()) + EPS
PaPr = Pa / Pr
PaPr_dB = 10.0 * math.log10(PaPr + EPS)

# ------------------------------------------------------------
# 8) Print summary
# ------------------------------------------------------------
if TARGET_MODE.lower() == "noise":
    tgt_label = "noise(shuffled)"
elif TARGET_MODE.lower() == "object":
    tgt_label = f"aug_{TARGET_RAW_SELECT}" if "TARGET_RAW_SELECT" in globals() else "aug(selected)"
else:
    tgt_label = TARGET_MODE

print("\n--- FINAL METRICS (ROI FOR OBJECT MODE) ---")
print(f"MSE(A,T)      : {mse_AT:.4e}")
print(f"MSE(A,C)      : {mse_AC:.4e}")
print(f"NCC(A,T)      : {ncc_AT:.4f}")
print(f"NCC(A,C)      : {ncc_AC:.4f}")
print(f"PSNR(A,C)     : {psnr_AC:.2f} dB")
print(f"SSIM(A,C)     : {ssim_AC:.4f}")
print(f"PSNR(A,T)     : {psnr_AT:.2f} dB")
print(f"SSIM(A,T)     : {ssim_AT:.4f}")
print(f"Pa/Pr         : {PaPr:.4e} ({PaPr_dB:.2f} dB)")

# ------------------------------------------------------------
# 9) Visualization with ROI boxes
# ------------------------------------------------------------
diff2d = adv2d - cln2d

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

ax = axes[0, 0]
im = ax.imshow(cln2d, cmap="gray", origin=ORIGIN)
ax.set_title(f"Clean {MODEL_TYPE} [{ATTACK_MODE}]")
ax.axis("off")
draw_roi_box(ax, roi_box_clean, color="r", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[0, 1]
im = ax.imshow(tgt2d, cmap="gray", origin=ORIGIN)
ax.set_title(f"Target ({tgt_label})")
ax.axis("off")
draw_roi_box(ax, roi_box_target, color="g", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 0]
im = ax.imshow(adv2d, cmap="gray", origin=ORIGIN)
ax.set_title("Adversarial (Attacked)")
ax.axis("off")
draw_roi_box(ax, roi_box_clean, color="r", lw=2)
draw_roi_box(ax, roi_box_target, color="g", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1, 1]
im = ax.imshow(diff2d, cmap="gray", origin=ORIGIN)
ax.set_title("Diff (A - C)")
ax.axis("off")
draw_roi_box(ax, roi_box_clean, color="r", lw=2)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()